# 💻 Módulo 04 - Práctica: Pipeline End-to-End

## Proyecto Integrador Completo

```python
import mlflow
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# 🟤 1. DATA INGESTION (Bronze)
print("🟤 Paso 1: Ingestar datos (Bronze)")
np.random.seed(42)
n_samples = 5000

data = pd.DataFrame({
    'customer_id': range(n_samples),
    'age': np.random.randint(18, 70, n_samples),
    'tenure_months': np.random.randint(1, 60, n_samples),
    'monthly_spend': np.random.uniform(20, 200, n_samples),
    'num_products': np.random.randint(1, 5, n_samples)
})

# Simular churn (target)
data['churn'] = ((data['tenure_months'] < 12) & (data['monthly_spend'] < 50)).astype(int)

print(f"✅ Datos generados: {len(data)} clientes")
print(f"Tasa de churn: {data['churn'].mean():.2%}")

# 🥈 2. DATA PROCESSING (Silver)
print("\n🥈 Paso 2: Procesar datos (Silver)")

# Limpieza y validación
data_clean = data[
    (data['age'] >= 18) & 
    (data['age'] <= 100) &
    (data['monthly_spend'] > 0)
].copy()

print(f"✅ Datos limpios: {len(data_clean)} registros")

# 🥇 3. FEATURE ENGINEERING (Gold)
print("\n🥇 Paso 3: Feature Engineering (Gold)")

# Crear features derivadas
data_clean['spend_per_product'] = data_clean['monthly_spend'] / data_clean['num_products']
data_clean['is_young'] = (data_clean['age'] < 30).astype(int)
data_clean['is_new_customer'] = (data_clean['tenure_months'] < 6).astype(int)

features = ['age', 'tenure_months', 'monthly_spend', 'num_products', 
            'spend_per_product', 'is_young', 'is_new_customer']

print(f"✅ Features creadas: {len(features)}")

# 🏋️‍♂️ 4. TRAINING
print("\n🏋️‍♂️ Paso 4: Entrenar modelo")

X = data_clean[features]
y = data_clean['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# MLflow experiment tracking
mlflow.set_experiment("/Users/shared/churn_prediction_e2e")

with mlflow.start_run(run_name="rf_baseline"):
    # Entrenar
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42
    )
    model.fit(X_train, y_train)
    
    # Evaluar
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Log
    mlflow.log_params({
        "n_estimators": 100,
        "max_depth": 10,
        "n_features": len(features)
    })
    
    mlflow.log_metrics({
        "accuracy": accuracy,
        "f1_score": f1
    })
    
    mlflow.sklearn.log_model(model, "model")
    
    print(f"✅ Modelo entrenado")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  F1-Score: {f1:.4f}")

# 🚀 5. VALIDATION
print("\n🚀 Paso 5: Validar modelo")

min_accuracy_threshold = 0.75

if accuracy >= min_accuracy_threshold:
    print(f"✅ Modelo pasa validación (accuracy >= {min_accuracy_threshold})")
    deploy_approved = True
else:
    print(f"❌ Modelo no pasa validación (accuracy < {min_accuracy_threshold})")
    deploy_approved = False

# 📦 6. REGISTRY
if deploy_approved:
    print("\n📦 Paso 6: Registrar modelo")
    
    # En producción real:
    # model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"
    # mlflow.register_model(model_uri, "churn_predictor")
    
    print("✅ Modelo registrado en MLflow Registry")
    print("  Listo para deployment a Staging")

# 📊 7. MONITORING SETUP
print("\n📊 Paso 7: Configurar monitoreo")

monitoring_config = {
    "drift_detection": "PSI",
    "threshold": 0.2,
    "frequency": "daily",
    "alerts": ["email", "slack"]
}

print("✅ Monitoreo configurado:")
for key, value in monitoring_config.items():
    print(f"  {key}: {value}")

# ✅ 8. RESUMEN
print("\n" + "="*50)
print("🎉 PIPELINE END-TO-END COMPLETADO")
print("="*50)

pipeline_summary = pd.DataFrame({
    'Paso': [
        '1. Data Ingestion',
        '2. Data Processing',
        '3. Feature Engineering',
        '4. Training',
        '5. Validation',
        '6. Registry',
        '7. Monitoring'
    ],
    'Estado': ['✅']*7,
    'Output': [
        f"{len(data)} records",
        f"{len(data_clean)} records",
        f"{len(features)} features",
        f"Accuracy: {accuracy:.4f}",
        "Passed" if deploy_approved else "Failed",
        "Model registered" if deploy_approved else "Skipped",
        "Configured"
    ]
})

print("\n" + pipeline_summary.to_string(index=False))

print("\n💡 Próximos pasos en producción:")
print("  1. Deploy to Staging endpoint")
print("  2. Run A/B test (90% old, 10% new)")
print("  3. Monitor performance for 1 week")
print("  4. Promote to Production if metrics hold")
print("  5. Setup automatic retraining (weekly)")

print("\n✅ ¡Proyecto integrador completado!")
```

---

**Universidad del Aconcagua 🇦🇷**

🎉 **¡Felicitaciones! Unidad 04 completada.**